In [1]:
import fsspec
import xarray as xr
import pandas as pd 
import os
from datetime import date
import datetime
import regionmask
import geopandas as gpd
import scipy.stats as stats
from ERA5_functions import *

In [17]:
ds1 = xr.open_mfdataset('/data/keeling/a/rytam2/a/iema_output/variables_202602080250.nc') #t2m, skt
ds2 = xr.open_mfdataset('/data/keeling/a/rytam2/a/iema_output/variables_202602150221.nc') # u10, v10, dp

In [18]:
t2m = ds1.t2m.sel(time=ds.time.dt.year <= 2025)
u10 = ds2.u10.sel(time=ds.time.dt.year <= 2025)
v10 = ds2.v10.sel(time=ds.time.dt.year <= 2025)

fpath = '/data/keeling/a/rytam2/a/iema_output/arcgis_toprocess/' #for output to arcgis

In [70]:
sfc_wind,_= wind_tot(u10,v10) # tier 1 
wchill = wind_chill(t2m, sfc_wind) # tier 2

/data/keeling/a/rytam2/climstat/local/done/ERA5_functions.py:190: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  wind_chill_set = xr.combine_by_coords((wind_chill,T_F,sfcWind_mph))


In [85]:
## daily windchill 
wchill_dmean = wchill.resample(time='1D').mean(dim='time').rename('DAILY_MEAN')
wchill_dmin = wchill.resample(time='1D').min(dim='time').rename('DAILY_MIN') 
wchill_dmax = wchill.resample(time='1D').min(dim='time').rename('DAILY_MAX') 
wchill_filtered = wchill_dmean.where(wchill_dmean > 50, np.nan).rename('DAILY_MEAN(>50=NAN)')

daily_compiled = xr.merge([wchill_dmean, wchill_filtered, wchill_dmin,wchill_dmax])
daily_compiled.to_netcdf(fpath+'dailystats_'+'windchill'+'_by_county_2016-2025'+'.nc')

In [60]:
## tot hours in month 
wchill_f = wchill
hourlymask = wchill_f<-20
grouped_hourly = hourlymask.groupby(['time.year', 'time.month']).sum(dim='time').rename('TOTAL_HOURS_LESS_N20F').stack(time=('year', 'month'))#.to_dataframe().drop(columns=['county', 'year', 'month'])

## tot days in month 
wchill_dmin = wchill_f.resample(time='1D').min(dim='time') # daily minimum windchill
dailymask = wchill_dmin<-20
grouped_daily = dailymask.groupby(['time.year', 'time.month']).sum(dim='time').rename('TOTAL_DAYS_LESS_N20F').stack(time=('year', 'month'))#.to_dataframe().drop(columns=['county', 'year', 'month'])

# monthly_compiled = xr.merge([grouped_hourly,grouped_daily]).reset_index('time')
# monthly_compiled['time']=pd.date_range(start='2016-01-01', periods=10*12, freq='MS')
monthly_compiled = xr.merge([grouped_hourly, grouped_daily]).reset_index('time')
monthly_compiled['time'] = pd.to_datetime(
    monthly_compiled['year'].astype(str) + '-' + monthly_compiled['month'].astype(str).str.zfill(2) + '-01'
)
monthly_compiled = monthly_compiled.drop_vars(['year', 'month'])

# print(monthly_compiled['TOTAL_HOURS_LESS_N20F'].isel(lat=12,lon=23,time=slice(23,38)).values)
# print(monthly_compiled['TOTAL_DAYS_LESS_N20F'].isel(lat=12,lon=23,time=slice(23,38)).values)
# monthly_compiled.to_netcdf(fpath+'monthlystats_'+'windchill'+'_by_county_2016-2025'+'.nc')

In [63]:
### tot hours in a year (calendar year)
grouped_hourly_yr = hourlymask.groupby(['time.year']).sum(dim='time').rename('TOTAL_HOURS_LESS_N20F')
grouped_daily_yr = dailymask.groupby(['time.year']).sum(dim='time').rename('TOTAL_DAYS_LESS_N20F')

yearly_compiled = xr.merge([grouped_hourly_yr,grouped_daily_yr])

# yearly_compiled.to_netcdf(fpath+'yearlystats_'+'windchill'+'_by_county_2016-2025'+'.nc')

In [64]:
### summary 
grouped_hourly_smry = hourlymask.sum(dim='time').rename('TOTAL_HOURS_LESS_N20F')#.stack(time=('year', 'month'))
grouped_daily_smry = dailymask.sum(dim='time').rename('TOTAL_DAYS_LESS_N20F')#.stack(time=('year', 'month'))

summary_compiled = xr.merge([grouped_hourly_smry,grouped_daily_smry])
# summary_compiled.to_netcdf(fpath+'summarystats_'+'windchill'+'_by_county_2016-2025'+'.nc')

In [29]:
gdf = gpd.read_file('/data/keeling/a/rytam2/climstat/local/county_zc_shpfile/zc/tl_2025_us_zcta520.shp')
gdf["ZCTA5CE20_int"] = gdf["ZCTA5CE20"].astype(int)
zcta_il = gdf[(gdf["ZCTA5CE20_int"] >= 60001) & (gdf["ZCTA5CE20_int"]<= 62999)]
zcta_il = zcta_il.reset_index(drop=True)

In [32]:
zcta_il["ZCTA5CE20_int"].unique()

array([62914, 62828, 61928, ..., 61020, 60645, 62951], shape=(1396,))

In [7]:
grouped_hourly_smry

<xarray.DataArray 'TOTAL_HOURS_LESS_N20F' (lat: 29, lon: 27)> Size: 6kB
dask.array<sum-aggregate, shape=(29, 27), dtype=int64, chunksize=(29, 27), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>

In [ ]:
zcta_il

In [8]:
ds =  grouped_hourly_smry
gdf = zcta_il 
variable_name=None


if isinstance(ds, xr.Dataset):
    if variable_name is None:
        raise ValueError("variable_name must be specified for xarray.Dataset")
    var_data = ds[variable_name]
else:
    var_data = ds

In [12]:
# Get coordinate arrays
lats = var_data.lat.values
lons = var_data.lon.values
# Normalize longitudes to -180 to 180
var_data = var_data.assign_coords(lon=(((lons + 180) % 360) - 180))

# years = var_data.year.values


In [ ]:
gdf = gdf.to_crs('EPSG:4326')

In [ ]:
# Ensure shapefile has correct CRS
if gdf.crs is None:
    gdf = gdf.set_crs('EPSG:4326')
else:
    gdf = gdf.to_crs('EPSG:4326')

# Create zone_id if it doesn't exist
if 'zone_id' not in gdf.columns:
    gdf['zone_id'] = np.arange(len(gdf))

# Calculate grid cell dimensions (assuming regular grid)
lat_res = np.abs(np.diff(lats).mean())
lon_res = np.abs(np.diff(lons).mean())

KeyboardInterrupt: 

Exception ignored in: 'pyproj._context.pyproj_log_function'
Traceback (most recent call last):
  File "/data/keeling/a/rytam2/miniconda3/envs/iema/lib/python3.13/logging/__init__.py", line 1498, in debug
    def debug(self, msg, *args, **kwargs):
KeyboardInterrupt: 


In [ ]:

# # Create grid cells as polygons
# grid_cells = []
# grid_lons = []
# grid_lats = []
# grid_lat_idx = []
# grid_lon_idx = []

# for i, lat in enumerate(lats):
#     for j, lon in enumerate(lons):
#         # Create bounding box for each grid cell
#         # Grid cell is centered on the coordinate point
#         min_lon = lon - lon_res / 2
#         max_lon = lon + lon_res / 2
#         min_lat = lat - lat_res / 2
#         max_lat = lat + lat_res / 2
        
#         cell_polygon = box(min_lon, min_lat, max_lon, max_lat)
#         grid_cells.append(cell_polygon)
#         grid_lons.append(lon)
#         grid_lats.append(lat)
#         grid_lat_idx.append(i)
#         grid_lon_idx.append(j)

# # Create GeoDataFrame of grid cells
# grid_gdf = gpd.GeoDataFrame({
#     'geometry': grid_cells,
#     'grid_lon': grid_lons,
#     'grid_lat': grid_lats,
#     'lat_idx': grid_lat_idx,
#     'lon_idx': grid_lon_idx
# }, crs='EPSG:4326')

# print(f"Created {len(grid_gdf)} grid cells")
# print(f"Grid resolution: {lat_res:.4f}° lat x {lon_res:.4f}° lon")

# # Spatial join: find which grid cells intersect with each polygon
# # This assigns zone_id to grid cells that fall within polygons
# joined = gpd.sjoin(grid_gdf, gdf[['zone_id', 'geometry']], 
#                    how='inner', predicate='intersects')

# print(f"Found {len(joined)} grid cell-polygon intersections")
# print(f"Polygons with at least one grid cell: {joined['zone_id'].nunique()}")

# # If a grid cell intersects multiple polygons, it will appear multiple times
# # Group by grid cell and zone to remove duplicates
# joined = joined.drop_duplicates(subset=['lat_idx', 'lon_idx', 'zone_id'])

# # Initialize results list
# results = []

# # For each year, calculate zonal statistics
# for year_idx, year in enumerate(years):
#     print(f"Processing year {year}...")
    
#     # Extract data for this year
#     year_data = var_data.isel(year=year_idx).values
    
#     # Add values to joined dataframe
#     joined['value'] = joined.apply(
#         lambda row: year_data[row['lat_idx'], row['lon_idx']], 
#         axis=1
#     )
    
#     # Calculate statistics for each zone
#     zonal_stats = joined.groupby('zone_id')['value'].agg([
#         ('mean', 'mean'),
#         ('std', 'std'),
#         ('min', 'min'),
#         ('max', 'max'),
#         ('count', 'count'),
#         ('sum', 'sum')
#     ]).reset_index()
    
#     zonal_stats['year'] = year
#     results.append(zonal_stats)

# # Combine all years
# results_df = pd.concat(results, ignore_index=True)

# # Reorder columns
# results_df = results_df[['year', 'zone_id', 'mean', 'std', 'min', 'max', 'count', 'sum']]

# # Add information about polygons with no data
# all_zone_ids = set(gdf['zone_id'].values)
# zones_with_data = set(results_df['zone_id'].unique())
# zones_without_data = all_zone_ids - zones_with_data
# print(len(zones_with_data))
# if zones_without_data:
#     #print(f"\nWarning: {len(zones_without_data)} polygons have no intersecting grid cells:")
#     #print(f"Zone IDs: {sorted(zones_without_data)}")
    
#     # Add rows for zones without data (with NaN values)
#     for zone_id in zones_without_data:
#         for year in years:
#             results_df = pd.concat([results_df, pd.DataFrame({
#                 'year': [year],
#                 'zone_id': [zone_id],
#                 'mean': [np.nan],
#                 'std': [np.nan],
#                 'min': [np.nan],
#                 'max': [np.nan],
#                 'count': [0],
#                 'sum': [np.nan]
#             })], ignore_index=True)

# results_df = results_df.sort_values(['year', 'zone_id']).reset_index(drop=True)

In [ ]:
import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

def calculate_zonal_statistics_gis_style(ds, gdf, variable_name=None):
    """
    Calculate zonal statistics (like GIS Zonal Statistics tool) for each polygon.
    Each polygon gets statistics from all grid cells that intersect with it.
    
    Parameters:
    -----------
    ds : xarray.Dataset or xarray.DataArray
        Dataset containing the variable to analyze
    gdf : geopandas.GeoDataFrame
        GeoDataFrame with zone polygons
    variable_name : str, optional
        Name of the variable in the dataset (not used if ds is DataArray)
    
    Returns:
    --------
    pd.DataFrame : Zonal statistics for each zone and year
    gpd.GeoDataFrame : Grid cells with their assigned zones
    """
    
    # Extract the variable (handle both Dataset and DataArray)
    if isinstance(ds, xr.Dataset):
        if variable_name is None:
            raise ValueError("variable_name must be specified for xarray.Dataset")
        var_data = ds[variable_name]
    else:
        var_data = ds
    
    # Get coordinate arrays
    lats = var_data.lat.values
    lons = var_data.lon.values
    # Normalize longitudes to -180 to 180
    lons = (((lons + 180) % 360) - 180)
    
    # Update the dataset with normalized longitudes
    var_data = var_data.assign_coords(lon=lons)
    
    years = var_data.year.values
    
    # Ensure shapefile has correct CRS
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    else:
        gdf = gdf.to_crs('EPSG:4326')
    
    # Create zone_id if it doesn't exist
    if 'zone_id' not in gdf.columns:
        gdf['zone_id'] = np.arange(len(gdf))
    
    # Calculate grid cell dimensions (assuming regular grid)
    lat_res = np.abs(np.diff(lats).mean())
    lon_res = np.abs(np.diff(lons).mean())
    
    # Create grid cells as polygons
    grid_cells = []
    grid_lons = []
    grid_lats = []
    grid_lat_idx = []
    grid_lon_idx = []
    
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            # Create bounding box for each grid cell
            # Grid cell is centered on the coordinate point
            min_lon = lon - lon_res / 2
            max_lon = lon + lon_res / 2
            min_lat = lat - lat_res / 2
            max_lat = lat + lat_res / 2
            
            cell_polygon = box(min_lon, min_lat, max_lon, max_lat)
            grid_cells.append(cell_polygon)
            grid_lons.append(lon)
            grid_lats.append(lat)
            grid_lat_idx.append(i)
            grid_lon_idx.append(j)
    
    # Create GeoDataFrame of grid cells
    grid_gdf = gpd.GeoDataFrame({
        'geometry': grid_cells,
        'grid_lon': grid_lons,
        'grid_lat': grid_lats,
        'lat_idx': grid_lat_idx,
        'lon_idx': grid_lon_idx
    }, crs='EPSG:4326')
    
    print(f"Created {len(grid_gdf)} grid cells")
    print(f"Grid resolution: {lat_res:.4f}° lat x {lon_res:.4f}° lon")
    
    # Spatial join: find which grid cells intersect with each polygon
    # This assigns zone_id to grid cells that fall within polygons
    joined = gpd.sjoin(grid_gdf, gdf[['zone_id', 'geometry']], 
                       how='inner', predicate='intersects')
    
    print(f"Found {len(joined)} grid cell-polygon intersections")
    print(f"Polygons with at least one grid cell: {joined['zone_id'].nunique()}")
    
    # If a grid cell intersects multiple polygons, it will appear multiple times
    # Group by grid cell and zone to remove duplicates
    joined = joined.drop_duplicates(subset=['lat_idx', 'lon_idx', 'zone_id'])
    
    # Initialize results list
    results = []
    
    # For each year, calculate zonal statistics
    for year_idx, year in enumerate(years):
        print(f"Processing year {year}...")
        
        # Extract data for this year
        year_data = var_data.isel(year=year_idx).values
        
        # Add values to joined dataframe
        joined['value'] = joined.apply(
            lambda row: year_data[row['lat_idx'], row['lon_idx']], 
            axis=1
        )
        
        # Calculate statistics for each zone
        zonal_stats = joined.groupby('zone_id')['value'].agg([
            ('mean', 'mean'),
            ('std', 'std'),
            ('min', 'min'),
            ('max', 'max'),
            ('count', 'count'),
            ('sum', 'sum')
        ]).reset_index()
        
        zonal_stats['year'] = year
        results.append(zonal_stats)
    
    # Combine all years
    results_df = pd.concat(results, ignore_index=True)
    
    # Reorder columns
    results_df = results_df[['year', 'zone_id', 'mean', 'std', 'min', 'max', 'count', 'sum']]
    
    # Add information about polygons with no data
    all_zone_ids = set(gdf['zone_id'].values)
    zones_with_data = set(results_df['zone_id'].unique())
    zones_without_data = all_zone_ids - zones_with_data
    print(len(zones_with_data))
    if zones_without_data:
        #print(f"\nWarning: {len(zones_without_data)} polygons have no intersecting grid cells:")
        #print(f"Zone IDs: {sorted(zones_without_data)}")
        
        # Add rows for zones without data (with NaN values)
        for zone_id in zones_without_data:
            for year in years:
                results_df = pd.concat([results_df, pd.DataFrame({
                    'year': [year],
                    'zone_id': [zone_id],
                    'mean': [np.nan],
                    'std': [np.nan],
                    'min': [np.nan],
                    'max': [np.nan],
                    'count': [0],
                    'sum': [np.nan]
                })], ignore_index=True)
    
    results_df = results_df.sort_values(['year', 'zone_id']).reset_index(drop=True)
    
    return results_df, joined


# Example usage
print("Starting zonal statistics calculation...")

# Calculate zonal statistics
zonal_stats, grid_zone_mapping = calculate_zonal_statistics_gis_style(
    t2m_hrs_below_0[0], 
    zcta_il
)

# Display results
print("\n" + "="*80)
print("ZONAL STATISTICS SUMMARY")
print("="*80)
print(f"\nTotal polygons in shapefile: {len(gdf)}")
print(f"Polygons with data: {zonal_stats[zonal_stats['count'] > 0]['zone_id'].nunique()}")
print(f"Total year-zone combinations: {len(zonal_stats)}")

print("\nFirst 20 rows of results:")
print(zonal_stats.head(20))

print("\nSample statistics by zone (first year):")
first_year = zonal_stats['year'].min()
print(zonal_stats[zonal_stats['year'] == first_year][['zone_id', 'mean', 'count']].head(10))

# Create summary pivot table
pivot_table = zonal_stats.pivot(index='year', columns='zone_id', values='mean')
print("\nPivot Table (Mean by Year and Zone):")
print(pivot_table)

# Summary statistics across all zones
print("\nGrid cells per polygon statistics:")
cells_per_zone = grid_zone_mapping.groupby('zone_id').size()
print(f"Min cells per polygon: {cells_per_zone.min()}")
print(f"Max cells per polygon: {cells_per_zone.max()}")
print(f"Mean cells per polygon: {cells_per_zone.mean():.2f}")
print(f"Median cells per polygon: {cells_per_zone.median():.0f}")

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Time series of zonal means (sample of zones)
unique_zones = zonal_stats[zonal_stats['count'] > 0]['zone_id'].unique()
n_zones_to_plot = min(15, len(unique_zones))

for i, zone_id in enumerate(unique_zones[:n_zones_to_plot]):
    zone_data = zonal_stats[zonal_stats['zone_id'] == zone_id]
    axes[0].plot(zone_data['year'], zone_data['mean'], 
                marker='o', label=f'Zone {int(zone_id)}', 
                alpha=0.7, linewidth=2)

axes[0].set_xlabel('Year', fontsize=12)
axes[0].set_ylabel('Mean Hours Below 0°C', fontsize=12)
axes[0].set_title(f'Zonal Mean Time Series (showing {n_zones_to_plot} of {len(unique_zones)} zones)', 
                 fontsize=13, fontweight='bold')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=1)
axes[0].grid(True, alpha=0.3)

# Plot 2: Heatmap of zonal means
if len(pivot_table) > 0 and len(pivot_table.columns) > 0:
    # Show all zones with data
    cols_to_plot = [c for c in pivot_table.columns if zonal_stats[zonal_stats['zone_id']==c]['count'].max() > 0]
    cols_to_plot = cols_to_plot[:min(30, len(cols_to_plot))]  # Limit for visibility
    pivot_subset = pivot_table[cols_to_plot]
    
    im = axes[1].imshow(pivot_subset.T, aspect='auto', cmap='RdYlBu_r', 
                       interpolation='nearest')
    axes[1].set_xlabel('Year', fontsize=12)
    axes[1].set_ylabel('Zone ID', fontsize=12)
    axes[1].set_title(f'Heatmap of Zonal Means (showing {len(cols_to_plot)} zones)', 
                     fontsize=13, fontweight='bold')
    axes[1].set_xticks(range(0, len(pivot_subset), max(1, len(pivot_subset)//10)))
    axes[1].set_xticklabels(pivot_subset.index[::max(1, len(pivot_subset)//10)], rotation=45)
    plt.colorbar(im, ax=axes[1], label='Hours Below 0°C')

# Plot 3: Distribution of grid cells per zone
axes[2].hist(cells_per_zone.values, bins=30, edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Number of Grid Cells per Polygon', fontsize=12)
axes[2].set_ylabel('Number of Polygons', fontsize=12)
axes[2].set_title('Distribution of Grid Cells per Polygon', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].axvline(cells_per_zone.mean(), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {cells_per_zone.mean():.1f}')
axes[2].axvline(cells_per_zone.median(), color='green', linestyle='--', 
               linewidth=2, label=f'Median: {cells_per_zone.median():.0f}')
axes[2].legend()

plt.tight_layout()
#plt.savefig('zonal_statistics_gis_style.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("Analysis complete! Plot saved as 'zonal_statistics_gis_style.png'")
print("="*80)
